# Llama grounding judge: eight-example calibration
1. Open https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct while signed in; request/accept access and wait until approved.
2. Create/use a read token from that same Hugging Face account. In Colab open Secrets (key icon), add HF_TOKEN, and enable notebook access. Do not paste the token into cells or chat.
3. Let Drive sync. Select T4 GPU and Run all. If access is uncertain, run the first three cells on CPU first, then switch to T4 and rerun all.
4. Send the final Latest report. Stop after these eight calls.

This is a separate Llama judge in four-bit inference, not Qwen self-evaluation. No automatic model fallback or retry. Synthetic assistant-written calibration answers are not human ground truth. Success here does not establish research-level judge reliability.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Disconnect and delete runtime, reconnect, then complete Google authorisation. No inference has started.') from exc
if not (ROOT/'Files/phm_judge_calibration.py').is_file():
    raise FileNotFoundError(f'Wait for Drive sync or correct the project path: {ROOT}. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_judge_calibration.py': '48b0dc185b83f6fa0f0040599a6348234ebe69951969872d982931386d9e41e9', 'test_phm_judge_calibration.py': '2cf90544f0a3ae8ae1f4f5029cf34cabc9132f6bf53260acbcb5c66d3a0bf952', 'phm_grounding_review.py': 'bfd2dcc11ff8c8b63775f077a213bc743cf44f65b7af82211df2528fa7ae1140', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9', 'phm_run_reporting.py': 'fa54efc1c88773ca86e0e1a2343ff83dc34fff5a3d2545dd0a8c9e51415da2a2', 'requirements-qwen-colab.txt': '0290713affb7b64776db6d09da7735ef3abbf12c38cf511c473abb0ec76f4226', 'experiment_config.json': '563acb189f79b8f6335900a164502d01895409bee9747ee6a0f70073c63e9b3b'}
PACK_HASHES = {'calibration_requests.jsonl': '9f0a53317e5ee720514e1519836d2528757c05d8e6960ce106f850b788ea0e54', 'calibration_answers_DO_NOT_SEND_TO_JUDGE.jsonl': '8e4110dd029a5db006792a61b3fb5c5a696f15cf814ab2b0f3cbb196927445ae'}
for name, expected in EXPECTED.items():
    assert hashlib.sha256((ROOT/'Files'/name).read_bytes()).hexdigest() == expected, f'Not fully synced: {name}'
for name, expected in PACK_HASHES.items():
    assert hashlib.sha256((ROOT/'Outputs/reviews/grounding_v1'/name).read_bytes()).hexdigest() == expected, f'Not fully synced: {name}'
print('Code and calibration pack verified.')


Mounted at /content/drive
Code and calibration pack verified.


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-qwen-colab.txt', 'lm-format-enforcer==0.11.3'], check=True)
from huggingface_hub import hf_hub_download, get_token
from google.colab import userdata
# The secret is used only for Hugging Face access; it is never printed or saved to Drive.
try:
    token = userdata.get('HF_TOKEN')
except Exception:
    token = get_token()
if not token:
    raise RuntimeError('Add a Hugging Face read token as the Colab secret HF_TOKEN and enable notebook access. Your account must also have access to meta-llama/Llama-3.1-8B-Instruct.')
os.environ['HF_TOKEN'] = token
try:
    hf_hub_download('meta-llama/Llama-3.1-8B-Instruct', 'config.json', revision='0e9e39f249a16976918f6564b8830bc894c89659', token=token)
except Exception:
    raise RuntimeError('Llama access check failed. Confirm access approval on its Hugging Face model page and use a token from that account. If access is approved, check the network and token permissions. No model weights were loaded.') from None
print('Pinned Llama checkpoint access confirmed. Token not logged.')


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

Pinned Llama checkpoint access confirmed. Token not logged.


In [3]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', 'test_phm_judge_calibration.py', '-v'], check=True)
subprocess.run([sys.executable, 'Files/phm_judge_calibration.py', '--root', str(ROOT), '--prepare-only'], check=True)
print('Eight calibration requests prepared. Expected answers are excluded from model prompts.')


Eight calibration requests prepared. Expected answers are excluded from model prompts.


In [4]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun all cells.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'Files/phm_judge_calibration.py', '--root', str(ROOT)], check=True)
latest = json.loads((ROOT/'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
print('Latest report:', latest)
print('STOP: let Drive sync and send this report. Calibration only; no diagnosis or PHM explanation judging was run.')


GPU: Tesla T4
Latest report: {'run_id': '20261122T020232_Llama_grounding_calibration_2c29e299', 'run_directory': 'Outputs/runs/20261122T020232_Llama_grounding_calibration_2c29e299', 'status': 'completed_with_issues', 'updated_utc': '2026-11-22T02:13:29.606703+00:00'}
STOP: let Drive sync and send this report. Calibration only; no diagnosis or PHM explanation judging was run.
